# Lab 8 — Change of Basis
**Linear Algebra · UATX**

If $\mathcal{B} = \{b_1,\ldots,b_n\}$ is a basis of $\mathbb{R}^n$ and $P = [b_1 \mid \cdots \mid b_n]$, then:
- The $\mathcal{B}$-coordinates of $v$ are $[v]_{\mathcal{B}} = P^{-1}v$.
- A linear map with standard-basis matrix $A$ has $\mathcal{B}$-basis matrix $A' = P^{-1}AP$.
- $A$ and $A'$ are called **similar**; they represent the same map in different bases.

**Tasks**
1. Implement `coords(v, B)` and convert vectors between bases.
2. Compute $A' = P^{-1}AP$ and compare it to $A$.
3. Diagonalise a $2 \times 2$ matrix via its eigenbasis; verify $P^{-1}AP = \Lambda$.
4. Use $A^k = P\Lambda^k P^{-1}$ to compute matrix powers; derive the Binet formula for Fibonacci.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(1)

## §1  Coordinates in a New Basis

In [ ]:
def coords(v, B):
    """
    Coordinates of v in the basis B.
    B : n x n matrix whose columns are basis vectors.
    Returns [v]_B = B^{-1} v.
    """
    return np.linalg.solve(B, v)

def from_coords(c, B):
    """Recover v from coordinates c in basis B: v = B c."""
    return B @ c


# --- Example: B = {(1,1), (1,-1)} / sqrt(2)  (orthonormal basis of R^2) ---
B = np.array([[1., 1.], [1., -1.]]) / np.sqrt(2)

v = np.array([3., 1.])
c = coords(v, B)
print('v in standard basis:', v)
print('v in basis B:       ', np.round(c, 4))
print('Reconstruct v from coords:', np.round(from_coords(c, B), 4))
print('Match:', np.allclose(from_coords(c, B), v))

In [ ]:
# Visualise: v decomposed in both bases
fig, ax = plt.subplots(figsize=(6, 6))
colors = ['steelblue', 'tomato']

# Standard basis vectors and v
for e, lbl, col in [(np.array([1.,0.]),'$e_1$','gray'), (np.array([0.,1.]),'$e_2$','gray')]:
    ax.quiver(0,0,*e, color=col, angles='xy', scale_units='xy', scale=1, alpha=0.4)
    ax.text(*(e*1.1), lbl, fontsize=10, color=col)

# Basis B vectors
for i, (bi, col) in enumerate(zip(B.T, ['steelblue','tomato'])):
    ax.quiver(0,0,*bi, color=col, angles='xy', scale_units='xy', scale=1, lw=2)
    ax.text(*(bi*1.15), f'$b_{i+1}$', fontsize=11, color=col)

# Vector v decomposed in B
ax.quiver(0,0,*v, color='black', angles='xy', scale_units='xy', scale=1, lw=2.5)
ax.text(*(v+0.1), f'$v=({v[0]},{v[1]})$\n$[v]_B=({c[0]:.2f},{c[1]:.2f})$', fontsize=9)

ax.set_xlim(-1, 4); ax.set_ylim(-2, 2); ax.set_aspect('equal'); ax.grid(True)
ax.set_title('Vector $v$ in standard and $\\mathcal{B}$ bases')
plt.tight_layout(); plt.show()

## §2  Matrix of a Map in a New Basis

If $T$ has standard matrix $A$, its matrix in basis $\mathcal{B}$ (columns of $P$) is $A' = P^{-1}AP$.

The map is the *same*; only the coordinate system changes.

In [ ]:
# A = rotation by pi/4
theta = np.pi / 4
A = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

# Change-of-basis matrix P (columns = new basis vectors)
P = np.array([[1., 0.], [1., 1.]])   # non-standard basis
P_inv = np.linalg.inv(P)

A_prime = P_inv @ A @ P
print('A  (standard basis) ='); print(np.round(A, 4))
print('\nA\' = P^{-1}AP (B-basis) ='); print(np.round(A_prime, 4))

# Verify: applying A to v in standard coords == converting, applying A', converting back
v = np.array([2., 1.])
lhs = A @ v
rhs = P @ (A_prime @ P_inv @ v)
print('\nA v                  =', np.round(lhs, 4))
print('P A\' P^{-1} v         =', np.round(rhs, 4))
print('Same result:', np.allclose(lhs, rhs))

## §3  Diagonalisation via the Eigenbasis

If $A$ has $n$ linearly independent eigenvectors $v_1,\ldots,v_n$ with eigenvalues $\lambda_1,\ldots,\lambda_n$, then $P = [v_1|\cdots|v_n]$ diagonalises $A$:
$$P^{-1}AP = \Lambda = \mathrm{diag}(\lambda_1,\ldots,\lambda_n).$$

In [ ]:
A_diag = np.array([[3., 1.],
                   [0., 2.]], dtype=float)

# Find eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(A_diag)
print('Eigenvalues:', eigenvalues)
print('Eigenvectors (columns):')
print(eigenvectors)

P_eig = eigenvectors   # change-of-basis matrix
Lambda = P_eig @ A_diag @ np.linalg.inv(P_eig)    # wrong order!
Lambda = np.linalg.inv(P_eig) @ A_diag @ P_eig
print('\nP^{-1} A P =')
print(np.round(Lambda, 8))
print('Is diagonal:', np.allclose(Lambda, np.diag(np.diag(Lambda))))
print('Diagonal entries = eigenvalues?', np.allclose(np.sort(np.diag(Lambda)), np.sort(eigenvalues)))

## §4  Matrix Powers and the Fibonacci Sequence

If $A = P\Lambda P^{-1}$, then $A^k = P \Lambda^k P^{-1}$ where $\Lambda^k = \mathrm{diag}(\lambda_1^k, \ldots, \lambda_n^k)$. This is dramatically faster than repeated matrix multiplication.

**Fibonacci:** $F_0=0, F_1=1, F_{n+1}=F_n+F_{n-1}$. We have
$$\begin{pmatrix}F_{n+1}\\F_n\end{pmatrix} = M^n \begin{pmatrix}1\\0\end{pmatrix}, \quad M = \begin{pmatrix}1&1\\1&0\end{pmatrix}.$$

In [ ]:
def matrix_power_via_diag(A, k):
    """
    Compute A^k using diagonalisation A = P Lambda P^{-1}.
    Only valid for diagonalisable matrices.
    """
    lam, P = np.linalg.eig(A)
    P_inv = np.linalg.inv(P)
    Lambda_k = np.diag(lam**k)
    return (P @ Lambda_k @ P_inv).real


M = np.array([[1., 1.], [1., 0.]])

# Check against np.linalg.matrix_power
for k in [5, 10, 20, 30]:
    Mk_diag = matrix_power_via_diag(M, k)
    Mk_np   = np.linalg.matrix_power(M, k)
    F_k = int(round(Mk_np[1, 0]))   # F_k
    F_k1= int(round(Mk_np[0, 0]))   # F_{k+1}
    print(f'k={k:2d}:  F_{k} = {F_k:8d},  F_{k+1} = {F_k1:8d},  diag match: {np.allclose(Mk_diag, Mk_np, atol=0.5)}')

In [ ]:
# Derive the Binet formula analytically
# M has eigenvalues phi = (1+sqrt(5))/2 (golden ratio) and psi = (1-sqrt(5))/2
phi = (1 + np.sqrt(5)) / 2
psi = (1 - np.sqrt(5)) / 2

print(f'Eigenvalues of M: phi = {phi:.6f}, psi = {psi:.6f}')
print(f'phi + psi = {phi+psi:.0f}  (should be 1 = trace(M))')
print(f'phi * psi = {phi*psi:.0f}  (should be -1 = det(M))')

# Binet formula: F_n = (phi^n - psi^n) / sqrt(5)
def binet(n):
    return (phi**n - psi**n) / np.sqrt(5)

print('\nBinet formula vs recurrence:')
F = [0, 1]
for i in range(2, 15):
    F.append(F[-1] + F[-2])
for n in range(1, 15):
    F_binet = round(binet(n))
    print(f'  F_{n:2d} = {F[n]:4d}  (Binet: {binet(n):.4f} -> rounded: {F_binet:4d})  match: {F_binet == F[n]}')

In [ ]:
# Trace and determinant are preserved under similarity
A_rand = np.random.randn(4, 4)
P_rand = np.random.randn(4, 4)
A_prime_rand = np.linalg.inv(P_rand) @ A_rand @ P_rand

print('Similarity invariants:')
print(f'  tr(A)  = {np.trace(A_rand):.6f},   tr(A\') = {np.trace(A_prime_rand):.6f}')
print(f'  det(A) = {np.linalg.det(A_rand):.6f},   det(A\') = {np.linalg.det(A_prime_rand):.6f}')
print(f'  Eigenvalues of A:  ', np.round(sorted(np.linalg.eigvals(A_rand).real), 4))
print(f'  Eigenvalues of A\': ', np.round(sorted(np.linalg.eigvals(A_prime_rand).real), 4))